In [1]:
import sys
import os
import io
import pandas as pd
sys.path.append(os.path.abspath(".."))
from churn.datos import cargar_datos, reporte_calidad, tasa_por_grupo, limpiar_nombres_columnas

In [2]:
pd.set_option('display.max_rows', 100)

In [3]:
normal = pd.read_csv('../data/dataset.csv', sep=';', decimal=',')
crudo = pd.read_csv('../data/dataset.csv', sep=';', decimal=',', keep_default_na=False)

nulos_normal = normal.isna().sum()  # NaN por columna (una Series)
vacios_crudo = (crudo == '').sum() # celdas vacías por columna

diferencia = nulos_normal - vacios_crudo # se restan columna con columna
diferencia[diferencia != 0] # solo las columnas donde no coincide

/tmp/ipykernel_69939/3893560373.py:2: DtypeWarning: Columns (75: phones, 76: models, 98: eqpdays) have mixed types. Specify dtype option on import or set low_memory=False.
  crudo = pd.read_csv('../data/dataset.csv', sep=';', decimal=',', keep_default_na=False)


hnd_webcap    10188
dtype: int64

In [4]:
print("Conteo de valores en 'hnd_webcap' para el DataFrame crudo:")
print(crudo['hnd_webcap'].value_counts())
print()
print("Conteo de valores en 'hnd_webcap' para el DataFrame normal:")
print(normal['hnd_webcap'].value_counts(dropna=False))

Conteo de valores en 'hnd_webcap' para el DataFrame crudo:
hnd_webcap
WCMB    75733
WC      13843
NA      10188
UNKW      235
            1
Name: count, dtype: int64

Conteo de valores en 'hnd_webcap' para el DataFrame normal:
hnd_webcap
WCMB    75733
WC      13843
NaN     10189
UNKW      235
Name: count, dtype: int64


In [5]:
print("Promedio de churn agrupado por hnd_webcap en el DataFrame normal:")
print(normal.groupby('hnd_webcap', dropna=False)["churn"].mean())

print()

print("Promedio de churn agrupado por hnd_webcap en el DataFrame crudo:")
print(crudo.groupby('hnd_webcap')["churn"].mean())

Promedio de churn agrupado por hnd_webcap en el DataFrame normal:
hnd_webcap
UNKW    0.365957
WC      0.563028
WCMB    0.470799
NaN     0.591520
Name: churn, dtype: float64

Promedio de churn agrupado por hnd_webcap en el DataFrame crudo:
hnd_webcap
        0.000000
NA      0.591578
UNKW    0.365957
WC      0.563028
WCMB    0.470799
Name: churn, dtype: float64


In [6]:
diccionario_datos = pd.read_csv("../data/data_descriptions.csv", sep=";")
diccionario_datos.head()

,Variable,Significado
0,Rev_Mean,Mean monthly revenue (charge amount)
1,mou_Mean,Mean number of monthly minutes of use
2,totmrc_Mean,Mean total monthly recurring charge
3,da_Mean,Mean number of directory assisted calls
4,ovrmou_Mean,Mean overage minutes of use


In [7]:
limpiar_nombres_columnas(diccionario_datos['Variable'])

0             rev_mean
1             mou_mean
2          totmrc_mean
3              da_mean
4          ovrmou_mean
5          ovrrev_mean
6          vceovr_mean
7          datovr_mean
8            roam_mean
9           change_mou
10          change_rev
11       drop_vce_mean
12       drop_dat_mean
13       blck_vce_mean
14       blck_dat_mean
15       unan_vce_mean
16       unan_dat_mean
17       plcd_vce_mean
18       plcd_dat_mean
19       recv_vce_mean
20       recv_sms_mean
21       comp_vce_mean
22       comp_dat_mean
23       custcare_mean
24       ccrndmou_mean
25         cc_mou_mean
26       inonemin_mean
27       threeway_mean
28       mou_cvce_mean
29       mou_cdat_mean
30       mou_rvce_mean
31     owylis_vce_mean
32     mouowylisv_mean
33     iwylis_vce_mean
34     mouiwylisv_mean
35       peak_vce_mean
36       peak_dat_mean
37       mou_peav_mean
38       mou_pead_mean
39        opk_vce_mean
40        opk_dat_mean
41       mou_opkv_mean
42       mou_opkd_mean
43       dr

In [8]:
diccionario_datos['Variable'] = limpiar_nombres_columnas(diccionario_datos['Variable'])

desc_por_clave = dict(zip(diccionario_datos['Variable'], diccionario_datos['Significado']))

In [9]:
df = cargar_datos("../data/dataset.csv")

print(df.dtypes.value_counts())

float64    69
str        21
int64      10
Name: count, dtype: int64


In [10]:
descripcion = {col: desc_por_clave.get(col.lower()) for col in df.columns}

sin_descripcion = [col for col, d in descripcion.items() if d is None]

print(sin_descripcion)

[]


In [11]:
display(reporte_calidad(df))

,dtype,pct_nulos,n_unicos,ejemplo
numbcars,float64,49.366,3,3.0
dwllsize,str,38.308,15,A
HHstatin,str,37.923,6,C
ownrent,str,33.706,2,O
dwlltype,str,31.909,2,S
lor,float64,30.190,16,15.0
income,float64,25.436,9,4.0
adults,float64,23.019,6,1.0
infobase,str,22.079,2,M
prizm_social_one,str,7.388,5,S


In [12]:
# Contar valores negativos en columnas numéricas
valores_negativos = (df.select_dtypes(include=['number']) < 0).sum()
valores_negativos[valores_negativos > 0].sort_values()

avg6rev            3
rev_Mean           5
totmrc_Mean       23
eqpdays          133
change_mou     53651
change_rev     56150
dtype: int64

In [13]:
# descripcion de columna con valores negativos
for col in valores_negativos[valores_negativos > 0].sort_values().index:
    print(f"{col}: {desc_por_clave.get(col.lower())}")

avg6rev: Average monthly revenue over the previous six months
rev_Mean: Mean monthly revenue (charge amount)
totmrc_Mean: Mean total monthly recurring charge
eqpdays: Number of days (age) of current mobile phone
change_mou: Percentage change in monthly minutes of use vs previous three month average
change_rev: Percentage change in monthly revenue vs previous three month average


In [14]:
def tasa_por_grupo_desc(df: pd.DataFrame, grupo: pd.Series) -> None:
    print(f"Número de clientes y tasa de churn por {grupo.name}:")
    display(tasa_por_grupo(df, grupo))
    print(f"{grupo.name}: {desc_por_clave.get(grupo.name.lower())}")

In [15]:
print(f"churn promedio: {df['churn'].mean()}")

churn promedio: 0.49562


In [16]:
tasa_por_grupo_desc(df, df['hnd_webcap'])

Número de clientes y tasa de churn por hnd_webcap:


,n_clientes,tasa_churn
hnd_webcap,,
NA,10188,0.591578
WC,13843,0.563028
WCMB,75733,0.470799
UNKW,235,0.365957
NaN,1,0.000000


hnd_webcap: Handset web capability


El grupo `NA` tiene mayor tasa de churn con un 59.16%, 9.6 puntos porcentuales por encima del promedio (49.56%) , por lo tanto es el más riesgoso.
Por otro lado, considerando el número de clientes, el grupo `WCMB` es el que mayor número de clientes tiene y con una tasa de churn nada baja (47%) por lo que este grupo representa un mayor impacto con una reducción más grande de ingresos por cancelación.

In [17]:
tasa_por_grupo_desc(df, df['refurb_new'])   

Número de clientes y tasa de churn por refurb_new:


,n_clientes,tasa_churn
refurb_new,,
R,14406,0.534361
N,85593,0.489105
NaN,1,0.000000


refurb_new: Handset: refurbished or new


Aquí el grupo de mayor riesgo es el de dispositivos reacondicionados (refurbished) con un 53.44% en la tasa de churn lo que es 3.88 puntos porcentuales por encima del promedio (49.56%).

In [18]:
segmento_meses = pd.cut(df["months"], bins=[0, 11, 24, df["months"].max()])
tasa_por_grupo_desc(df, segmento_meses)

Número de clientes y tasa de churn por months:


,n_clientes,tasa_churn
months,,
"(11, 24]",48901,0.530235
"(24, 61]",24689,0.493702
"(0, 11]",26410,0.433321


months: Total number of months in service


Aquí el grupo con mayor riesgo son los clientes con una antigüedad mayor a 11 meses y menor a 24 meses, con una tasa de churn del 53.02% y siendo además este grupo el de mayor número de clientes. Este grupo está un 3.46 puntos porcentuales por encima del promedio (49.56%).
El patron en la tasa de churn por meses de antiguedad es bajo -> alto -> bajo. El grupo de 0-11 meses está 6.2 puntos por debajo del promedio. En telefonía, los clientes nuevos suelen tener un contrato típicamente de 12 meses y no pueden irse sin penalización. Cuando el contrato vence, el riesgo se dispara. Esto es una hipótesis pues no se tienen datos de fecha de fin de contrato, pero siguiendo esta hiótesis se podría hacer una recomendación concreta: contactar al cliente antes de que venza su contrato.